In [27]:
import pandas as pd
import os
import sys
import numpy as np

In [28]:
base_dir = os.path.dirname(os.path.dirname(os.getcwd()))
data_dir = os.path.join(base_dir, 'data', 'processed_data', 'House2_full.csv')

In [29]:
sys.path.append(base_dir)
from src.tools.window_shifter import WindowShifter

In [30]:
data_type_apps = {
    'Appliance1': np.int32,    
    'Appliance3': np.int32,    
    'Appliance4': np.int32,    
    'Appliance5': np.int32,   
    'Appliance6': np.int32,   
    'Appliance7': np.int32,   
    'Appliance8': np.int32,   
    'Appliance9': np.int32,  
}

In [38]:
df = pd.read_csv(data_dir, dtype = data_type_apps)

In [39]:
app_cols = ['Appliance1', 'Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6', 'Appliance7', 'Appliance8', 'Appliance9']

In [40]:
invalid_mask_train = df['Aggregate'] <= df[app_cols].sum(axis=1)
df.loc[invalid_mask_train, app_cols] = np.nan
df.loc[invalid_mask_train, 'Aggregate'] = np.nan
df[app_cols] = df[app_cols].ffill()
df[app_cols] = df[app_cols].fillna(0)
df['Aggregate'] = df['Aggregate'].ffill()
df['Aggregate'] = df['Aggregate'].fillna(0)

In [41]:
df_time_to_date = pd.to_datetime(df['Time'], format='%Y-%m-%d %H:%M:%S')
df['dom'] = df_time_to_date.dt.day

In [42]:
class WindowShifterForS2swa():
    def shift(df, n):
        tmp = pd.DataFrame()
        df_keep = df[['Aggregate']]
        df_app = df.drop(columns = ['Time', 'Unix', 'Aggregate']) 
        for i in range(n):
            tmp = pd.concat([tmp, df_keep.shift(i).rename(columns = {'Aggregate': f'Aggregate_t{i}'})], axis = 1)
        df_with_nan = pd.concat([tmp, df_app], axis = 1)
        nan_index = df_with_nan.isnull().sum(axis = 1)[df_with_nan.isnull().sum(axis = 1)>0].index
        df = df_with_nan.drop(nan_index)
        return df

In [43]:
df = WindowShifterForS2swa.shift(df, 300)

In [44]:
data_dir_shifted = os.path.join(base_dir, 'data', 'processed_data', 'House2_full_shifted.csv')
df.to_csv(data_dir_shifted, index = False)